In [ ]:
import pickle as pkl
import re
import toml

In [ ]:
def parse_attributes(desc):
    attrs = {}
    for part in desc.split(";"):
        part = part.strip()
        if not part or "=" not in part:
            continue
        k, v = part.split("=", 1)
        attrs[k.strip().lower()] = v.strip()
    return attrs


def read_fasta(path):
    header = None
    seq_chunks = []

    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            if line.startswith(">"):
                if header is not None:
                    yield header, "".join(seq_chunks)
                header = line[1:]
                seq_chunks = []
            else:
                seq_chunks.append(line)

        if header is not None:
            yield header, "".join(seq_chunks)


def convert_label(order, superfamily, lhs_to_rhs, rhs_labels):
    order = (order or "").strip()
    superfamily = (superfamily or "").strip()

    # 1. exact final Terrier label
    if order and superfamily:
        candidate = f"{order}/{superfamily}"
        if candidate in rhs_labels:
            return candidate, "exact_rhs_order_superfamily"

    # 2. map order if it exists on TOML LHS
    if order in lhs_to_rhs:
        return lhs_to_rhs[order], "lhs_order"

    # 3. map superfamily if it exists on TOML LHS
    if superfamily in lhs_to_rhs:
        return lhs_to_rhs[superfamily], "lhs_superfamily"

    # 4. otherwise drop
    return None, "unmapped"


def main():
    lhs_to_rhs = toml.load(MAPPING_FILE)
    rhs_labels = set(lhs_to_rhs.values())

    result = {}
    unmapped = {}
    stats = {
        "total": 0,
        "mapped": 0,
        "unmapped": 0,
        "exact_rhs_order_superfamily": 0,
        "lhs_order": 0,
        "lhs_superfamily": 0,
    }

    for header, sequence in read_fasta(INPUT_FASTA):
        sequence=sequence.lower()
        if len(sequence) < 80:
            continue
        
        stats["total"] += 1

        if " " in header:
            sequence_id, desc = header.split(None, 1)
        else:
            sequence_id, desc = header, ""
        

        attrs = parse_attributes(desc)
        order = attrs.get("order", "")
        superfamily = attrs.get("superfamily", "")

        mapped_label, strategy = convert_label(order, superfamily, lhs_to_rhs, rhs_labels)

        if mapped_label is None:
            unmapped[sequence] = {
                "sequence_id": sequence_id,
                "order": order,
                "superfamily": superfamily,
                "header": header,
            }
            stats["unmapped"] += 1
            continue

        result[sequence] = {
            "sequence_id": sequence_id,
            "mapped_label": mapped_label,
        }
        stats["mapped"] += 1
        stats[strategy] += 1

    with open(OUTPUT_PICKLE, "wb") as f:
        pkl.dump(result, f)

    with open(UNMAPPED_PICKLE, "wb") as f:
        pkl.dump(unmapped, f)

    print("Done")
    print(f"Total: {stats['total']}")
    print(f"Mapped: {stats['mapped']}")
    print(f"Unmapped/dropped: {stats['unmapped']}")
    print("Strategy breakdown:")
    for k in ["exact_rhs_order_superfamily", "lhs_order", "lhs_superfamily"]:
        print(f"  {k}: {stats[k]}")
    print(f"Saved mapped pickle: {OUTPUT_PICKLE}")
    print(f"Saved unmapped pickle: {UNMAPPED_PICKLE}")




In [19]:

INPUT_FASTA = "Whole_MnTEdb.fasta"
OUTPUT_PICKLE = "MnTEdb_terrier.pkl"
MAPPING_FILE = "repeatmasker_labels.toml"
UNMAPPED_PICKLE = "MnTEdb_unmapped.pkl"

if __name__ == "__main__":
    main()

Done
Total: 5925
Mapped: 5373
Unmapped/dropped: 552
Strategy breakdown:
  exact_rhs_order_superfamily: 2991
  lhs_order: 1011
  lhs_superfamily: 1371
Saved mapped pickle: MnTEdb_terrier.pkl
Saved unmapped pickle: MnTEdb_unmapped.pkl


In [22]:
with open(f"TERRIER_labellingsystems_alldatasets/{OUTPUT_PICKLE}", "rb") as f:
    unmapped_data = pkl.load(f)


for seq, info in list(unmapped_data.items())[:]:
    if len(info["mapped_label"].split("/")) ==1:
        print(f"Original label: {info.get('mapped_label', 'N/A')}")
        print(f"Sequence (first 50 chars): {seq[:50]}...")
        print("-" * 40)

Original label: LTR
Sequence (first 50 chars): tgtttgaaagagtgcgaggggagtctgagagtgaaagaccagctgaggaa...
----------------------------------------
Original label: LTR
Sequence (first 50 chars): tgtttgaaagagtgcgaggggagtctgagagtgaaagaccagctgaggaa...
----------------------------------------
Original label: LTR
Sequence (first 50 chars): tggcaacatttttttttattgaggtgttaaagggaaaatttttccttata...
----------------------------------------
Original label: LTR
Sequence (first 50 chars): tgtttgaaagagtgcgaggggagtctgagagtgaaagaccagctgaggaa...
----------------------------------------
Original label: LTR
Sequence (first 50 chars): tgagaaacgatgtgacaatcttgtttgaaagagtgcgaggggagtctgag...
----------------------------------------
Original label: LTR
Sequence (first 50 chars): tgtatcaaccacattttatccgcgaagtcagagagagaccgccgaggacc...
----------------------------------------
Original label: LTR
Sequence (first 50 chars): tgtatcaaccacattttatccgcgaagtcagagagagaccgccgaggacc...
----------------------------------------
Origin